### Imports

In [ ]:
from devsim import *
import matplotlib.pyplot as plt

## Device builder:

In [ ]:
# --- Device and Mesh Setup ---
device = "diode1d"
region = "Si"

# Use a robust direct solver
set_parameter(name="direct_solver", value="umfpack")

In [ ]:
# Create a 1D mesh for an ideal P-N diode
# The device is 100 microns long, with the junction at x=50 microns
create_1d_mesh(mesh="diode_mesh")
add_1d_mesh_line(mesh="diode_mesh", pos=0, ps=0.01, tag="p_contact_pos")
# pos is position and ps tels the point spacing.
# we make it more dense at the junction
add_1d_mesh_line(mesh="diode_mesh", pos=50, ps=0.001, tag="junction_pos")# Refine mesh at junction
add_1d_mesh_line(mesh="diode_mesh", pos=100, ps=0.01, tag="n_contact_pos")

In [ ]:
# Define contacts at the ends
add_1d_contact(mesh="diode_mesh", name="Anode", tag="p_contact_pos", material="metal")
add_1d_contact(mesh="diode_mesh", name="Cathode", tag="n_contact_pos", material="metal")

In [ ]:
# Define the silicon region
add_1d_region(mesh="diode_mesh", material="Si", region=region, tag1="p_contact_pos", tag2="n_contact_pos")

In [ ]:
# Finalize the mesh and create the device
finalize_mesh(mesh="diode_mesh")
create_device(mesh="diode_mesh", device=device)

## Material parameters

In [ ]:
# Set physical parameters for Silicon at 300K
# All units are in cm, V, s unless specified otherwise
set_parameter(device=device, region=region, name="Permittivity", value=11.8 * 8.854e-14) # F/cm
set_parameter(device=device, region=region, name="n_i", value=1.0e10) # Intrinsic carrier density (cm^-3)
set_parameter(device=device, region=region, name="T", value=300)
# Thermal voltage kT/q
Vt = 0.025851
set_parameter(device=device, name="V_t", value=Vt)

In [ ]:
# Mobilities and Einstein Relation
set_parameter(device=device, region=region, name="mu_n", value=400.0) # cm^2/(V*s)
set_parameter(device=device, region=region, name="mu_p", value=200.0)
set_parameter(device=device, region=region, name="Dn",
              value=get_parameter(device=device, region=region, name="mu_n")*Vt)
set_parameter(device=device, region=region, name="Dp", value=get_parameter(device=device, region=region, name="mu_p")*Vt)

In [ ]:
# Recombination parameters (SRH)
set_parameter(device=device, region=region, name="tau_n", value=1e-7) # Carrier lifetime (s)
set_parameter(device=device, region=region, name="tau_p", value=1e-7)

In [ ]:
# Optical Generation Parameters (Beer-Lambert Law)
# G(x) = G0 * exp(-alpha * x)
set_parameter(device=device, region=region, name="G0", value=0) # No light initially (dark current)
set_parameter(device=device, region=region, name="alpha", value=1e4) # Absorption coeff (cm^-1)

## Physics Equations Setup

In [ ]:
# Poisson Equation
create_poisson_equation(device=device, region=region)

In [ ]:
# Electron and Hole Continuity Equations
create_electron_equation(device=device, region=region, "Jn")
create_hole_equation(device=device, region=region, "Jp")

In [ ]:
# SRH Recombination Model
create_SRH_recombination(device=device, region=region)

In [ ]:
# Optical Generation Model
node_model(device=device, region=region, name="OpticalGeneration", equation="G0 * exp(-alpha * x)")
# Connect the optical model to the continuity equations
equation(device=device, region=region, name="ElectronContinuityEquation",
         node_model="OpticalGeneration", variable_update="positive")
equation(device=device, region=region, name="HoleContinuityEquation",
         node_model="OpticalGeneration", variable_update="positive")

In [ ]:
# ---- Contact Boundary Conditions ----
# We have two ohmic contacts: Anode (p-side) and Cathode (n-side)
for contact in get_contact_list(device=device):
    create_contact_poisson_equation(device=device, contact=contact)
    create_contact_electron_equation(device=device, contact=contact, "Jn")
    create_contact_hole_equation(device=device, contact=contact, "Jp")

## Solver


In [ ]:
# --- Solve for Equilibrium (0V bias) ---
print("Solving for equilibrium (0V bias)...")
solve(type="dc", absolute_error=1.0, relative_error=1e-12, maximum_iterations=30)
print("Equilibrium solved.")

In [ ]:
# --- Plot the equilibrium band diagram and carrier concentrations ---
x_coords = get_node_model_values(device=device, region=region, name="x")
potential = get_node_model_values(device=device, region=region, name="Potential")
electrons = get_node_model_values(device=device, region=region, name="Electrons")
holes = get_node_model_values(device=device, region=region, name="Holes")

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(x_coords, -potential, label='Ec, Ev')
plt.title('Equilibrium Band Diagram')
plt.xlabel('Position (microns)')
plt.ylabel('Energy (eV)')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.semilogy(x_coords, electrons, label='Electrons (n)')
plt.semilogy(x_coords, holes, label='Holes (p)')
plt.title('Carrier Concentrations')
plt.xlabel('Position (microns)')
plt.ylabel('Concentration (cm^-3)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

### DC Reverse Bias Sweep (Dark Current)

### DC Sweep with Optical Generation (Photocurrent)

### AC (Small-Signal) Analysis